# Task 1

In [2]:
pip install git+https://github.com/openai/whisper.git

Defaulting to user installation because normal site-packages is not writeable
  Cloning https://github.com/openai/whisper.git to /tmp/pip-req-build-qak9m7h4
  Running command git clone --filter=blob:none --quiet https://github.com/openai/whisper.git /tmp/pip-req-build-qak9m7h4
  Resolved https://github.com/openai/whisper.git to commit c0d2f624c09dc18e709e37c2ad90c039a4eb72a2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.7/69.7 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 6.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 8.5 MB/s eta 0:00:00a 0:00:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 1.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 4.4 MB/s 

In [15]:
!yt-dlp -o video.mp4 "https://www.youtube.com/watch?v=LPDnemFoqVk"

[youtube] Extracting URL: https://www.youtube.com/watch?v=LPDnemFoqVk
[youtube] LPDnemFoqVk: Downloading webpage
[youtube] LPDnemFoqVk: Downloading android sdkless player API JSON
[youtube] LPDnemFoqVk: Downloading web safari player API JSON
[youtube] LPDnemFoqVk: Downloading m3u8 information
[info] LPDnemFoqVk: Downloading 1 format(s): 399+251
[download] Sleeping 5.00 seconds as required by the site...
[download] Destination: video.mp4.f399.mp4
[download] 100% of    1.18GiB in 00:43:00 at 477.55KiB/s33m00:00
[download] Destination: video.mp4.f251.webm
[download] 100% of   87.24MiB in 00:05:36 at 265.66KiB/s33m00:00
[Merger] Merging formats into "video.mp4.webm"
Deleting original file video.mp4.f251.webm (pass -k to keep)
Deleting original file video.mp4.f399.mp4 (pass -k to keep)


In [8]:
# !whisper video.mp4 --model medium --device cuda --fp16 True
# !whisper video.mp4 --model medium --device cuda --fp16 True --output_format txt

input_file = "video.mp4"
!mkdir -p transcripts
!whisper video.mp4.webm --model medium --device cuda --output_format txt --output_dir transcripts
# !whisper video.mp4 --model medium --device cuda --fp16 True --output_format txt --output_dir transcripts

In [1]:
!ls transcripts

In [1]:
# !ffmpeg -i video.mp4.webm -vn -acodec pcm_s16le -ar 16000 -ac 1 audio.wav
# !whisper audio.wav --model medium --device cuda --output_format txt --output_dir transcripts

# !whisper audio.wav --model medium --device cuda --output_format txt --output_dir ./
import subprocess
import soundfile as sf
import os
from faster_whisper import WhisperModel

# --- CONFIGURATION ---
input_file = "audio.wav"
output_file = "transcript.txt"
temp_files = ["temp_part1.wav", "temp_part2.wav"]

# Use CPU/int8 for maximum stability on your system
# This avoids the GTX 1660 Ti Segfault issue completely
print("Loading model on CPU (Safe Mode)...")
model = WhisperModel("tiny", device="cpu", compute_type="int8")

# --- 1. GET DURATION ---
info = sf.info(input_file)
total_duration = info.duration
midpoint = total_duration / 2
print(f"Total Duration: {total_duration/60:.2f} mins")

# --- HELPER FUNCTION FOR FFMPEG ---
def slice_audio(start_time, duration, output_name):
    """Slices audio on disk using FFmpeg without loading RAM"""
    cmd = [
        "ffmpeg", "-y", "-v", "error", # Quiet mode
        "-ss", str(start_time),
        "-i", input_file,
        "-t", str(duration) if duration else str(total_duration), 
        "-c", "copy",
        output_name
    ]
    subprocess.run(cmd, check=True)

# --- MAIN EXECUTION ---
try:
    with open(output_file, "w", encoding="utf-8") as f:
        
        # --- PART 1 ---
        print(f"\n--- Processing Part 1 (0 to {midpoint/60:.2f} mins) ---")
        slice_audio(start_time=0, duration=midpoint, output_name=temp_files[0])
        
        segments, _ = model.transcribe(temp_files[0], beam_size=1)
        
        for segment in segments:
            # Format: [Start - End] Text
            timestamped_line = f"[{segment.start:.2f}s - {segment.end:.2f}s] {segment.text}"
            print(timestamped_line) # Print to console
            f.write(timestamped_line + "\n") # Write to file

        # --- PART 2 ---
        print(f"\n--- Processing Part 2 ({midpoint/60:.2f} mins to End) ---")
        # Duration=None means "until the end"
        slice_audio(start_time=midpoint, duration=None, output_name=temp_files[1])
        
        segments, _ = model.transcribe(temp_files[1], beam_size=1)
        
        for segment in segments:
            # CRITICAL: Add the midpoint offset to the timestamps
            real_start = segment.start + midpoint
            real_end = segment.end + midpoint
            
            timestamped_line = f"[{real_start:.2f}s - {real_end:.2f}s] {segment.text}"
            print(timestamped_line)
            f.write(timestamped_line + "\n")

    print(f"\nSuccess! saved to {output_file}")

finally:
    # Clean up temp files
    print("Cleaning up temporary files...")
    for tf in temp_files:
        if os.path.exists(tf):
            os.remove(tf)

/home/sheru/projects/ai-1/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading model on CPU (Safe Mode)...
Total Duration: 94.02 mins

--- Processing Part 1 (0 to 47.01 mins) ---
[0.00s - 1.36s]  I was not breaking news.
[1.36s - 3.40s]  There are different basketball team with a healthy
[3.40s - 4.76s]  whole LeBron James.
[4.76s - 9.00s]  15 games, all the 500, when he's healthy, three games
[9.00s - 10.60s]  under 500.
[10.60s - 13.12s]  Creeds offense for himself and his teammates.
[13.12s - 14.28s]  They need him.
[14.28s - 17.36s]  They all Jeff Steph Curry, what he's done over the past month
[17.36s - 20.16s]  and a half has been absolutely brilliant.
[20.16s - 22.96s]  He's as valuable as any player in this league.
[22.96s - 26.52s]  They play great defense and they have the ultimate home run
[26.52s - 27.72s]  hitter.
[27.72s - 31.08s]  Up beneath the team, it hasn't been easy this season.
[31.08s - 33.84s]  But both finished up the regular season strong.
[33.84s - 37.16s]  The Lakers ended with a five game winning streak.
[37.16s - 40.84s]  And 

In [2]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))

CUDA available: True
GPU count: 1
GPU Name: NVIDIA GeForce GTX 1660 Ti


In [3]:
import subprocess

def chat_with_ollama(model, message):
    # run Ollama model with input as stdin
    process = subprocess.Popen(
        ["ollama", "run", model],
        stdin=subprocess.PIPE,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )
    
    # send the user message to Ollama
    stdout, stderr = process.communicate(input=message)
    
    if process.returncode != 0:
        raise RuntimeError(stderr)
    
    return stdout.strip()

# Example usage
reply = chat_with_ollama("llama2", "Hello Ollama!")
print(reply)


*blinks* Uh, hi there! *giggles* I'm not sure if you're talking to me or the ol' lamp over there, but I'll play along! *bounces up and down* What can I do for you? Do you want to chat about something interesting? Or maybe you just wanted to say hi? *grins* Let me know!


In [4]:
class OllamaClient:
    def __init__(self, model="llama2"):
        self.model = model

    def chat(self, message):
        return chat_with_ollama(self.model, message)


In [5]:
client = OllamaClient(model="llama2")

In [9]:
# Load transcript
with open("transcript.txt", "r", encoding="utf-8") as f:
    transcript = f.read()

# Optional: split transcript into manageable chunks
CHUNK_SIZE = 1000  # characters per chunk
chunks = [transcript[i:i+CHUNK_SIZE] for i in range(0, len(transcript), CHUNK_SIZE)]
chunk_index = {i: chunk for i, chunk in enumerate(chunks)}


In [12]:
class OllamaQA:
    def __init__(self, client, chunk_index):
        self.client = client
        self.chunks = chunk_index

    def query(self, question):
        # Simple approach: concatenate all chunks (or top N) for context
        context = "\n".join(self.chunks.values())
        prompt = f"You are a basketball commentator assistant. Use only the following commentary to answer the question.\n\nCommentary:\n{context}\n\nQuestion: {question}\nAnswer:"
        
        # Ask the model
        response = self.client.chat(prompt)
        return response

In [13]:
client = OllamaClient(model="llama2")
qa_system = OllamaQA(client, chunk_index)

question = "Analyze the player that scored the most in this game"
answer = qa_system.query(question)

print("Q:", question)
print("A:", answer)

Q: Analyze the player that scored the most in this game
A: Based on the text provided, the player who scored the most in this game is LeBron James. The text states that James scored 22 points, which is the highest score mentioned in the passage.
